# Practical 5: Continuous Bag of Words (CBOW) Model

**Problem statement:** Implement the Continuous Bag of Words (CBOW) model using these stages:

- **(a) Data preparation**
- **(b) Generate training data**
- **(c) Train the model**
- **(d) Output**

This notebook uses *Alice's Adventures in Wonderland* from the NLTK Gutenberg corpus. Gensim's `Word2Vec` is configured with `sg=0`, which selects CBOW.

**Note:** The example in Stage (b) explicitly displays context words and their target word. Gensim generates the actual CBOW training examples internally when the Word2Vec model is trained.

## (a) Import required libraries

In [1]:
# (a) Install required libraries (run this cell if the packages are not installed)
%pip install -q gensim nltk

Note: you may need to restart the kernel to use updated packages.


In [1]:
# (a) Import required libraries

import re
import warnings
import numpy as np
import nltk
import gensim

from gensim.models import Word2Vec
from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

warnings.filterwarnings("ignore")

print("Gensim version:", gensim.__version__)

Gensim version: 4.4.0


## (a) Access the dataset and prepare the text

In [2]:
# (a) Download the NLTK resources required for the dataset and text processing

nltk.download("gutenberg", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

print("NLTK resources are ready.")

NLTK resources are ready.


In [3]:
# (a) Load Alice's Adventures in Wonderland

text = gutenberg.raw("carroll-alice.txt")

print("Characters in raw text:", len(text))

Characters in raw text: 144395


In [4]:
# (a) Split the text into sentences, clean it, lowercase it, and tokenize it

sentences = sent_tokenize(text)
stop_words = set(stopwords.words("english"))

data = []

for sentence in sentences:
    # Remove special characters, digits, and punctuation
    sentence = re.sub(r"[^A-Za-z\s]", " ", sentence)

    # Convert text to lowercase
    sentence = sentence.lower()

    # Tokenize and remove stopwords and single-character words
    words = [
        word for word in word_tokenize(sentence)
        if word not in stop_words and len(word) > 1
    ]

    if words:
        data.append(words)

print("Sentences after cleaning:", len(data))
print("Example processed sentence:", data[10][:15] if len(data) > 10 else data[0][:15])

Sentences after cleaning: 1590
Example processed sentence: ['first', 'tried', 'look', 'make', 'coming', 'dark', 'see', 'anything', 'looked', 'sides', 'well', 'noticed', 'filled', 'cupboards', 'book']


## (b) Generate CBOW training data

In [5]:
# (b) Demonstrate how CBOW context-target examples are formed
# The target word is surrounded by context words from the same processed sentence.
# Gensim creates the actual training examples internally during Word2Vec training.

window_size = 2
sample_sentence = next(sentence for sentence in data if len(sentence) >= 2 * window_size + 1)

print("Sample processed sentence:")
print(" ".join(sample_sentence[:15]))

print("\nCBOW context -> target examples (showing up to 5):")
shown = 0

for i in range(window_size, len(sample_sentence) - window_size):
    context = (
        sample_sentence[i - window_size:i]
        + sample_sentence[i + 1:i + window_size + 1]
    )
    target = sample_sentence[i]

    print(f"Context: {context} -> Target: {target}")
    shown += 1

    if shown == 5:
        break

Sample processed sentence:
alice adventures wonderland lewis carroll chapter

CBOW context -> target examples (showing up to 5):
Context: ['alice', 'adventures', 'lewis', 'carroll'] -> Target: wonderland
Context: ['adventures', 'wonderland', 'carroll', 'chapter'] -> Target: lewis


## (c) Train the CBOW model

In [6]:
# (c) Train Word2Vec using CBOW (sg=0)

model = Word2Vec(
    sentences=data,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,             # sg=0 selects CBOW
    workers=1,
    epochs=200,
    seed=1
)

print("CBOW model trained successfully!")
print("Vocabulary size:", len(model.wv))

CBOW model trained successfully!
Vocabulary size: 1313


## (d) Output: word vectors and similarity scores

In [7]:
# (d) Display the vector for 'alice' and similarities between selected words

if "alice" in model.wv:
    print("Vector for 'alice' (first 8 dimensions):")
    print(model.wv["alice"][:8].round(4))

if "alice" in model.wv and "queen" in model.wv:
    print("\nSimilarity ('alice', 'queen'):", round(float(model.wv.similarity("alice", "queen")), 4))

if "alice" in model.wv and "rabbit" in model.wv:
    print("Similarity ('alice', 'rabbit'):", round(float(model.wv.similarity("alice", "rabbit")), 4))

Vector for 'alice' (first 8 dimensions):
[-0.979  -0.1221  0.8731 -0.6154  0.2589  0.0142 -0.2142 -0.4761]

Similarity ('alice', 'queen'): 0.0951
Similarity ('alice', 'rabbit'): 0.1945


In [8]:
# (d) Display the five most similar words for selected vocabulary words

for word in ["alice", "queen", "king", "rabbit"]:
    print(f"\nTop-5 words most similar to '{word}':")

    if word in model.wv:
        for similar_word, score in model.wv.most_similar(word, topn=5):
            print(f"   {similar_word:<12} {score:.4f}")
    else:
        print("   Word not found in vocabulary.")


Top-5 words most similar to 'alice':
   absurd       0.3217
   conversation 0.3088
   temper       0.3059
   hard         0.2959
   chose        0.2766

Top-5 words most similar to 'queen':
   crimson      0.3900
   stamping     0.3837
   stopped      0.3707
   shouting     0.3679
   croquet      0.3568

Top-5 words most similar to 'king':
   verdict      0.4726
   crimson      0.4693
   executioner  0.4533
   executed     0.4489
   knave        0.4292

Top-5 words most similar to 'rabbit':
   directed     0.5696
   hedge        0.5315
   kid          0.5018
   started      0.4962
   stopped      0.4853


In [9]:
# (d) Display likely output words for example context words

contexts = [
    ["mad", "tea", "party"],
    ["white", "rabbit", "hole"]
]

for context in contexts:
    print(f"\nPredict output word for context {context}:")

    missing_words = [word for word in context if word not in model.wv]

    if missing_words:
        print("   Words missing from vocabulary:", missing_words)
        continue

    predictions = model.predict_output_word(context, topn=5)

    if predictions:
        for word, probability in predictions:
            print(f"   {word:<12} {probability:.4f}")
    else:
        print("   No predictions returned.")


Predict output word for context ['mad', 'tea', 'party']:
   tea          0.5692
   party        0.2926
   mad          0.0548
   pointing     0.0184
   table        0.0055

Predict output word for context ['white', 'rabbit', 'hole']:
   rabbit       0.9898
   started      0.0015
   hole         0.0015
   hedge        0.0009
   watched      0.0005


In [10]:
# (d) Save the trained CBOW model

model.save("cbow_word2vec.model")
print("Model saved as: cbow_word2vec.model")
print("Practical 5 completed successfully!")

Model saved as: cbow_word2vec.model
Practical 5 completed successfully!


## Result and conclusion

The CBOW model was implemented using Word2Vec on *Alice's Adventures in Wonderland*. The text was prepared and tokenized, context-target examples were demonstrated, and the model was trained with `sg=0` to select the CBOW architecture. The output includes word vectors, word-similarity scores, similar words, and example predicted output words. The trained model is saved as `cbow_word2vec.model`.

**Note:** Similarity scores and predicted words depend on the dataset, preprocessing, and training configuration.